<div style="background: linear-gradient(135deg, #1B3A6B 0%, #2563EB 100%); padding: 40px 36px 32px 36px; border-radius: 10px; color: white; font-family: Arial, sans-serif;">
  <div style="font-size: 13px; letter-spacing: 2px; opacity: 0.75; margin-bottom: 6px;">
    AI ETHICS MINOR &nbsp;•&nbsp; ARTI 459
  </div>
  <div style="font-size: 28px; font-weight: 700; line-height: 1.3; margin-bottom: 4px;">
    Group Assignment — AI Fairness Metrics
  </div>
  <div style="font-size: 14px; opacity: 0.80; margin-top: 6px;">
    Starter Notebook &nbsp;|&nbsp; Assessment Weight: 20 Marks
  </div>
</div>


## Student Information

| Field | Value |
|---|---|
| **Student 1 Name** | *Maisoon Mohammed Alabdullah* |
| **Student 1 ID** | *2240005251* |
| **Student 2 Name** | *Wajd Khald AlKhaldi* |
| **Student 2 ID** | *2240005544* |
| **Student 3 Name** | *Jolie Gamal Salama* |
| **Student 3 ID** | *2240004580* |
| **Student 4 Name** | *Manar Saleh Alghamdi* |
| **Student 4 ID** | *2240005128* |

> **Instructions:** Fill in the table above, then work through each Part in order.  
> Use the **code cells** for calculations and the **Markdown answer cells** for written responses.  
> Do not delete or reorder any cells — simply fill them in.

---
## Setup — Run This Cell First
Import libraries and define display helpers used throughout the notebook.


In [1]:
# ── Standard imports ──────────────────────────────────────────────────────────
import math
from IPython.display import display, HTML, Markdown

# ── Pretty-print a metric result ──────────────────────────────────────────────
def show_metric(name, value, target="≈ 0", passed=None, note=""):
    """Display a single fairness metric with colour-coded status."""
    if passed is None:
        colour = "#2563EB"       # blue  – neutral
    elif passed:
        colour = "#15803D"       # green – criterion met
    else:
        colour = "#DC2626"       # red   – criterion violated
    html = (
        f"<div style='font-family:monospace; background:#F3F6FB; border-left:4px solid {colour};"
        f"padding:10px 14px; margin:4px 0; border-radius:4px;'>"
        f"<b>{name}</b> &nbsp;=&nbsp; <span style='color:{colour}; font-size:1.1em;'>{value:.2f}</span>"
        f"&nbsp;&nbsp; <span style='color:#6B7280; font-size:0.88em;'>(target {target})</span>"
        + (f"<br><span style='color:#6B7280; font-size:0.85em; margin-top:2px;'>{note}</span>" if note else "")
        + "</div>"
    )
    display(HTML(html))

def show_section(title, colour="#1B3A6B"):
    display(HTML(
        f"<div style='background:{colour}; color:white; font-family:Arial,sans-serif;"
        f"font-size:15px; font-weight:700; padding:8px 14px; border-radius:6px; margin:18px 0 6px 0;'>"
        f"{title}</div>"
    ))

print("✅ Setup complete — you're ready to begin.")


✅ Setup complete — you're ready to begin.


---
## Notation Reminder (from the Handout)

| Symbol | Meaning |
|---|---|
| **A** | Protected attribute (e.g., gender, nationality) |
| **u** | Unprivileged group |
| **p** | Privileged group |
| **Ŷ = 1** | Positive model prediction (e.g., shortlisted, approved) |
| **Y = 1** | Ground-truth positive (truly qualified / creditworthy) |
| **TPR(g)** | True Positive Rate for group g = P(Ŷ=1 \| Y=1, A=g) |
| **FPR(g)** | False Positive Rate for group g = P(Ŷ=1 \| Y=0, A=g) |

### The Five Metrics at a Glance

| Metric | Formula | Ideal Value |
|---|---|---|
| Statistical Parity | P(Ŷ=1\|A=u) − P(Ŷ=1\|A=p) | ≈ 0 |
| Disparate Impact | P(Ŷ=1\|A=u) / P(Ŷ=1\|A=p) | ≈ 1 (≥ 0.8 rule of thumb) |
| Equal Opportunity | TPR(u) − TPR(p) | ≈ 0 |
| Equalized Odds | TPR(u) ≈ TPR(p) **and** FPR(u) ≈ FPR(p) | Both gaps ≈ 0 |
| Calibration | P(Y=1\|score=s, A=u) = P(Y=1\|score=s, A=p) | Equal across groups |


---
# Part 1 — Hiring Screening Model  *(10 Marks)*

**CLO 2.1** — Analyze ethical risks in AI systems using real-world case studies.

### Scenario
A technology company uses an automated model to shortlist candidates for software engineering roles.  
- **Protected attribute (A):** Gender  
- **Privileged group (p):** Male  
- **Unprivileged group (u):** Female  
- **Positive outcome (Ŷ = 1):** Shortlisted for interview

---
### Table 1 — Confusion Matrix Summary (n = 350 applicants)

| Metric | Male (A = p) | Female (A = u) |
|---|:---:|:---:|
| Total applicants | 200 | 150 |
| Predicted shortlisted (Ŷ = 1) | 80 | 45 |
| Actually qualified (Y = 1) | 100 | 90 |
| True Positives — TP | 70 | 36 |
| False Positives — FP | 10 | 9 |
| False Negatives — FN | 30 | 54 |
| True Negatives — TN | 90 | 51 |

> **Quick check:** TP + FP = predicted shortlisted, and TP + FN = actually qualified.  
> Male: 70+10 = 80 ✓ and 70+30 = 100 ✓ &nbsp;|&nbsp; Female: 36+9 = 45 ✓ and 36+54 = 90 ✓


### Part 1 — Data (Pre-loaded)

In [2]:
# ── Part 1 data — DO NOT EDIT ────────────────────────────────────────────────

# Male (privileged, p)
male = {
    "total":  200,
    "TP":      70,   # correctly shortlisted
    "FP":      10,   # wrongly shortlisted (not actually qualified)
    "FN":      30,   # missed (qualified but rejected)
    "TN":      90,   # correctly rejected
}

# Female (unprivileged, u)
female = {
    "total":  150,
    "TP":      36,
    "FP":       9,
    "FN":      54,
    "TN":      51,
}

# Derived counts (computed for you)
male["Y1"]   = male["TP"]   + male["FN"]    # actually qualified
male["Y0"]   = male["FP"]   + male["TN"]    # not qualified
male["Yhat1"]= male["TP"]   + male["FP"]    # predicted shortlisted
female["Y1"] = female["TP"] + female["FN"]
female["Y0"] = female["FP"] + female["TN"]
female["Yhat1"] = female["TP"] + female["FP"]

print("Part 1 data loaded.")
print(f"  Male   → qualified: {male['Y1']}, predicted+: {male['Yhat1']}, not qualified: {male['Y0']}")
print(f"  Female → qualified: {female['Y1']}, predicted+: {female['Yhat1']}, not qualified: {female['Y0']}")


Part 1 data loaded.
  Male   → qualified: 100, predicted+: 80, not qualified: 100
  Female → qualified: 90, predicted+: 45, not qualified: 60


---
### Q1.1 — Compute Fairness Metrics  *(6 Marks — 1.5 marks each)*

Calculate all four metrics below.  
**Show your working** by printing intermediate values (selection rates, TPR, FPR) before the final metric.


In [3]:
# ── Q1.1a · Statistical Parity ───────────────────────────────────────────────
# Formula: P(Ŷ=1|A=u) − P(Ŷ=1|A=p)
# Target:  value close to 0

show_section("Q1.1a — Statistical Parity")

# TODO: compute the selection rate for each group, then the difference
p_pos_female = female["Yhat1"] / female["total"]   # P(Ŷ=1 | A=female)
p_pos_male   = male["Yhat1"] / male["total"]   # P(Ŷ=1 | A=male)
stat_parity  = p_pos_female - p_pos_male   # difference

print(f"P(Ŷ=1 | Female) = {p_pos_female:.2f}")
print(f"P(Ŷ=1 | Male)   = {p_pos_male:.2f}")
show_metric("Statistical Parity (difference)", stat_parity, target="≈ 0")


P(Ŷ=1 | Female) = 0.30
P(Ŷ=1 | Male)   = 0.40


In [4]:
# ── Q1.1b · Disparate Impact ─────────────────────────────────────────────────
# Formula: P(Ŷ=1|A=u) / P(Ŷ=1|A=p)
# Target:  close to 1  |  Rule of thumb: flag if < 0.8

show_section("Q1.1b — Disparate Impact")

# TODO: compute the ratio (re-use p_pos_female and p_pos_male from above)
disparate_impact = p_pos_female / p_pos_male

passed_80_rule = disparate_impact >= 0.8
show_metric("Disparate Impact (ratio)", disparate_impact, target="≥ 0.8", passed=passed_80_rule,
            note=f"0.8 rule of thumb: {'MET ✓' if passed_80_rule else 'VIOLATED ✗'}")


In [5]:
# ── Q1.1c · Equal Opportunity ────────────────────────────────────────────────
# Formula: TPR(u) − TPR(p)   where   TPR(g) = TP / (TP + FN)
# Target:  value close to 0

show_section("Q1.1c — Equal Opportunity (TPR Parity)")

# TODO: compute TPR for each group, then the difference
tpr_female = female["TP"] / female["Y1"]   # TP_female / Y1_female
tpr_male   = male["TP"] / male["Y1"]   # TP_male   / Y1_male
eq_opp     = tpr_female - tpr_male   # difference

print(f"TPR(Female) = {tpr_female:.2f}")
print(f"TPR(Male)   = {tpr_male:.2f}")
show_metric("Equal Opportunity (TPR difference)", eq_opp, target="≈ 0")


TPR(Female) = 0.40
TPR(Male)   = 0.70


In [6]:
# ── Q1.1d · Equalized Odds ───────────────────────────────────────────────────
# Requires BOTH: TPR(u) ≈ TPR(p)  AND  FPR(u) ≈ FPR(p)
# FPR(g) = FP / (FP + TN)  =  FP / Y0
# Target:  both differences close to 0

show_section("Q1.1d — Equalized Odds (TPR + FPR Parity)")

# TODO: compute FPR for each group
fpr_female = female["FP"] / female["Y0"]   # FP_female / Y0_female
fpr_male   = male["FP"] / male["Y0"]   # FP_male   / Y0_male
fpr_gap    = fpr_female - fpr_male   # FPR difference

print(f"FPR(Female) = {fpr_female:.2f}")
print(f"FPR(Male)   = {fpr_male:.2f}")
show_metric("TPR gap (Equal Opportunity)", eq_opp, target="≈ 0")
show_metric("FPR gap", fpr_gap, target="≈ 0")

# TODO: set to True if BOTH gaps are close to 0, False otherwise
tolerance = 0.05   # "close to 0" = gap smaller than 0.05, same as what used in the last summary dashboard cell
eq_odds_satisfied = (abs(eq_opp) < tolerance) and (abs(fpr_gap) < tolerance)
print()
print("Equalized Odds satisfied?", "✓ YES" if eq_odds_satisfied else "✗ NO — at least one gap is non-trivial")


FPR(Female) = 0.15
FPR(Male)   = 0.10



Equalized Odds satisfied? ✗ NO — at least one gap is non-trivial


---
### Q1.2 — Interpret the Results  *(2 Marks)*

**Answer the following in the cell below (full sentences, ~150–200 words):**

1. Which fairness criteria are **violated** and which (if any) are **satisfied**? Cite your metric values.
2. Which group is **disadvantaged**? Use the numbers to support your claim.
3. Comment on what the **False Negative rate** means for qualified female applicants and the harm this causes.


**Q1.2 — Written Answer:**

**Criteria violated / satisfied:**  
None of the four criteria is fully satisfied.  
- **Statistical Parity** is −0.10 (selection rate 0.30 for females vs 0.40 for males), so it is not close to 0.
- **Disparate Impact** is 0.75, which is below the 0.8 rule of thumb, so there is a sign of negative impact.  
- **Equal Opportunity** is −0.30 (TPR 0.40 vs 0.70), which is the largest gap we found.
- **Equalized Odds** is also not satisfied. The FPR gap is 0.05 (0.15 vs 0.10), which is right at the tolerance limit so we treat it as borderline, but the TPR gap alone is enough to fail it.


**Which group is disadvantaged and why:**  
Female are disadvantaged. They are shortllisted less overall (30% vs 40%), and even among qualified applicants only 40% of women get shortlisted compared to 70% of men.

**False Negatives and harm:**  
The false negative rate for qualified women is 54/90 = 0.60, which is double the male (30/100 = 0.30). So 6 out of 10 qualified women are rejected before getting an interview. They lose job chances they deserve, and the company loses good engineers.


---
### Q1.3 — Metric Appropriateness  *(2 Marks)*

**Answer the following (~120–150 words):**

1. Why is **Equal Opportunity** a more appropriate metric than **Statistical Parity** in the hiring context?  
   *(Hint: reference the handout's caution that Statistical Parity does not use the true label Y.)*
2. Describe **one scenario** where Statistical Parity could give a **false sense of fairness**.


**Q1.3 — Written Answer:**

**Why Equal Opportunity is more appropriate here:**  
**Statistical Parity** only compares how many people from each group are shortlisted, it does not use the true label Y. In hiring, what is important is whether qualified people get a fair chance or not, so **Equal Opportunity** is better because it compares TPR among qualified applicants only (Y = 1). In our data women actually have a higher base rate (0.60 qualified vs 0.50 for men), yet qualified women have a TPR of only 0.40 vs 0.70. Equal Opportunity shows this clearly, while Statistical Parity only shows a 0.10 gap.

**A scenario where Statistical Parity misleads:**  
Asuume the model shortlists 40% of men and 40% of women, so **Statistical Parity** will be 0. But for women it mostly selects unqualified candidates and rejects the qualified ones, while for men it picks the qualified ones. The metric looks perfect, but qualified women are still treated unfairly.



---
# Part 2 — Loan Approval Model  *(6 Marks)*

**CLO 2.1** — Analyze ethical risks in AI systems using real-world case studies.

### Scenario
A financial institution uses an ML model to approve or reject personal loan applications.  
- **Protected attribute (A):** Nationality  
- **Privileged group (p):** Local residents  
- **Unprivileged group (u):** Expatriate residents  
- **Positive outcome (Ŷ = 1):** Loan approved

---
### Table 2 — Confusion Matrix Summary (n = 500 applicants)

| Metric | Local (A = p) | Expatriate (A = u) |
|---|:---:|:---:|
| Total applicants | 300 | 200 |
| Predicted approved (Ŷ = 1) | 180 | 100 |
| Actually creditworthy (Y = 1) | 210 | 160 |
| True Positives — TP | 160 | 88 |
| False Positives — FP | 20 | 12 |
| False Negatives — FN | 50 | 72 |
| True Negatives — TN | 70 | 28 |

> **Quick check:** Local: 160+20=180 ✓ and 160+50=210 ✓ &nbsp;|&nbsp; Expatriate: 88+12=100 ✓ and 88+72=160 ✓


### Part 2 — Data (Pre-loaded)

In [7]:
# ── Part 2 data — DO NOT EDIT ────────────────────────────────────────────────

# Local residents (privileged, p)
local = {
    "total":  300,
    "TP":     160,
    "FP":      20,
    "FN":      50,
    "TN":      70,
}

# Expatriate residents (unprivileged, u)
expat = {
    "total":  200,
    "TP":      88,
    "FP":      12,
    "FN":      72,
    "TN":      28,
}

# Derived counts
for g in [local, expat]:
    g["Y1"]    = g["TP"] + g["FN"]
    g["Y0"]    = g["FP"] + g["TN"]
    g["Yhat1"] = g["TP"] + g["FP"]

print("Part 2 data loaded.")
print(f"  Local    → creditworthy: {local['Y1']}, predicted+: {local['Yhat1']}, not creditworthy: {local['Y0']}")
print(f"  Expat    → creditworthy: {expat['Y1']}, predicted+: {expat['Yhat1']}, not creditworthy: {expat['Y0']}")


Part 2 data loaded.
  Local    → creditworthy: 210, predicted+: 180, not creditworthy: 90
  Expat    → creditworthy: 160, predicted+: 100, not creditworthy: 40


---
### Q2.1 — Compute All Four Fairness Metrics  *(4 Marks — 1 mark each)*

Use the same formulas as Part 1. Show intermediate values before each metric.


In [8]:
# ── Q2.1a · Statistical Parity ───────────────────────────────────────────────
show_section("Q2.1a — Statistical Parity", colour="#1B3A6B")

# TODO: compute selection rates and their difference
p_pos_expat  = expat["Yhat1"] / expat["total"]
p_pos_local  = local["Yhat1"] / local["total"]
stat_parity2 = p_pos_expat - p_pos_local

print(f"P(Ŷ=1 | Expat) = {p_pos_expat:.2f}")
print(f"P(Ŷ=1 | Local) = {p_pos_local:.2f}")
show_metric("Statistical Parity (difference)", stat_parity2, target="≈ 0")


P(Ŷ=1 | Expat) = 0.50
P(Ŷ=1 | Local) = 0.60


In [9]:
# ── Q2.1b · Disparate Impact ─────────────────────────────────────────────────
show_section("Q2.1b — Disparate Impact", colour="#1B3A6B")

# TODO: compute the ratio
disparate_impact2 = p_pos_expat / p_pos_local

passed2 = disparate_impact2 >= 0.8
show_metric("Disparate Impact (ratio)", disparate_impact2, target="≥ 0.8", passed=passed2,
            note=f"0.8 rule of thumb: {'MET ✓' if passed2 else 'VIOLATED ✗'}")


In [10]:
# ── Q2.1c · Equal Opportunity ────────────────────────────────────────────────
show_section("Q2.1c — Equal Opportunity (TPR Parity)", colour="#1B3A6B")

# TODO: compute TPR for each group, then the difference
tpr_expat = expat["TP"] / expat["Y1"]
tpr_local = local["TP"] / local["Y1"]
eq_opp2   = tpr_expat - tpr_local

print(f"TPR(Expat) = {tpr_expat:.2f}")
print(f"TPR(Local) = {tpr_local:.2f}")
show_metric("Equal Opportunity (TPR difference)", eq_opp2, target="≈ 0")


TPR(Expat) = 0.55
TPR(Local) = 0.76


In [11]:
# ── Q2.1d · Equalized Odds ───────────────────────────────────────────────────
show_section("Q2.1d — Equalized Odds", colour="#1B3A6B")

# TODO: compute FPR for each group
fpr_expat  = expat["FP"] / expat["Y0"]
fpr_local  = local["FP"] / local["Y0"]
fpr_gap2   = fpr_expat - fpr_local

print(f"FPR(Expat) = {fpr_expat:.2f}")
print(f"FPR(Local) = {fpr_local:.2f}")
show_metric("TPR gap (Equal Opportunity)", eq_opp2, target="≈ 0")
show_metric("FPR gap", fpr_gap2, target="≈ 0")

tolerance = 0.05   # same tolerance used in the summary dashboard
eq_odds2 = (abs(eq_opp2) < tolerance) and (abs(fpr_gap2) < tolerance)
print()
print("Equalized Odds satisfied?", "✓ YES" if eq_odds2 else "✗ NO")


FPR(Expat) = 0.30
FPR(Local) = 0.22



Equalized Odds satisfied? ✗ NO


---
### Q2.2 — Conflicting Metric Signals  *(2 Marks)*

**Answer all three points (~150–200 words total):**

1. The Disparate Impact ratio **meets** the 0.8 threshold, yet Equal Opportunity reveals a meaningful gap.  
   What does this inconsistency tell us about **relying on a single metric**?

2. Which metric is more **ethically meaningful** for loan approvals, and why?  
   Connect your answer to the concrete harm experienced by creditworthy expatriate applicants who are wrongly rejected.

3. Explain what **Calibration** means in the loan context, and why a lender might want to evaluate it *alongside* Equal Opportunity.


**Q2.2 — Written Answer:**

*(Replace this text with your answer.)*

**On the inconsistency between Disparate Impact and Equal Opportunity:**    
disparate Impact is 0.83, so the model passes the 80% rule and looks fair. But Equal Opportunity is −0.21 (TPR 0.55 for expats vs 0.76 for locals). Each metric measures something different (outcome rates vs error rates), so relying on only one can hide a real problem.

**Which metric is more ethically meaningful and why:**    
Since **Equal Opportunity** focuses on those who can genuinely repay, we believe it has greater significance for loans. Compared to 24% of residents, 72 out of 160 qualified foreigners (45%) were incorrectly denied credit. These applicants may turn to more expensive lenders after losing money they might need for housing or emergency. Disparate Impact ignores Y, therefore it is unable to see this.

**What Calibration means here and why it matters alongside Equal Opportunity:**    
The model gives each applicant a score for how likely they are to repay. Calibration means this score is equally honest for both groups, for example a score of 0.7 should mean a 70% chance of repaying for a local and for an expat. If expats' scores are too low, fewer of them pass the cut-off, which could explain their lower TPR. our table has no scores, so we can only explain it, not compute it.

---
# Part 3 — Responsible AI Proposal  *(4 Marks)*

**CLO 2.2** — Propose responsible AI solutions considering fairness, humanity and sustainability.

**Instructions:** Choose **one** of the two scenarios from Part 1 or Part 2.  
Write your response in the Markdown cells below (~300–400 words total across all three sub-questions).  
Use full sentences — not bullet points.


**Chosen Scenario:** Part 2 — Loan Approval


---
### Q3.1 — Bias Mitigation Strategy  *(2 Marks)*

Choose **one** mitigation strategy from the three categories covered in the lecture (L04):

| Category | Examples from AIF360 |
|---|---|
| **Pre-processing** | Reweighing, Disparate Impact Remover, Optimised Pre-processing |
| **In-processing** | Adversarial Debiasing, Prejudice Remover Regulariser |
| **Post-processing** | Equalized Odds Post-processing, Reject-Option Classification |

In your answer:
- Name and briefly describe the chosen strategy.
- Justify why it suits your scenario (what data/model access does it require?).
- State which fairness metric(s) it is most likely to improve.
- Note any **trade-offs** with overall model accuracy.


**Q3.1 — Written Answer:**

**Chosen strategy and description:**  
We choose **Equalized Odds Post-processing** from **AIF360**. It runs after training: using each group's predictions and true labels, it flips some decisions with a certain probability so that TPR and FPR become close for locals and expats.

**Justification for this scenario:**  
Targeting mistake rates makes sense because the loan model's primary issue is unequal error rates (TPR gap −0.21 and FPR gap 0.08), not only approval rates. The genuine repayment labels, the nationality attribute on a validation set, and the model's predictions are all that are required. If the model is purchased from a third-party provider, there is no need for retraining.

**Metrics it is likely to improve:**  
It should mainly improve Equal Opportunity and Equalized Odds, because the method directly equalizes TPR and FPR between groups.

**Trade-offs with accuracy:**  
Because it changes some of the model's decisions to close the gap, a few of them will be wrong, so overall accuracy drops a little. It also needs to know each applicant's nationality, which must be justified (see Q3.2).


---
### Q3.2 — SDAIA Fairness Principle Alignment  *(1 Mark)*

The SDAIA AI Ethics Principles (introduced in L04) state that fairness requires:
- Eliminating **bias, discrimination, and stigmatisation** across the AI lifecycle
- Using **just, non-biased, and objective** design standards
- Ensuring systems are **inclusive, diverse, and representative**

Explain how your proposed mitigation strategy directly addresses **at least one** of these elements.  
Connect it concretely to your scenario — do not speak in generalities.


**Q3.2 — Written Answer:**

**SDAIA principle element addressed:**  
Removing bias and discrimination by using objective standards. inn the Build and Validate stage, SDAIA says the final model's fairness metrics must stay within a set limit for sensitive features, they call it "champion model's fairness metrics"

**How the mitigation strategy implements this element in practice:**  
In our case the sensitive feature is nationality. Before the fix, 45% of creditworthy expats were rejected compared to only 24% of locals. The post-processing sets a clear limit: the TPR and FPR gaps must be less than 0.05. This way, a creditworthy expat gets almost the same chance of approval as a creditworthy local. SDAIA also says using sensitive data must be justified, so the bank would write down that nationality is used only to fix this bias.


---
### Q3.3 — Post-Deployment Monitoring Plan  *(1 Mark)*

Describe a brief monitoring plan for after the model goes live. Address:
- Which **fairness metric(s)** would you track and at what **frequency**?
- What **threshold or signal** would trigger a re-audit or model update?
- Why is **continuous monitoring necessary** even after a debiasing algorithm has been applied?


**Q3.3 — Written Answer:**

**Metrics to monitor and frequency:**  
every month we would check the:  
- TPR gap
-  FPR gap
-   Disparate Impact between expats and locals.  
For the TPR and FPR we need to know who actually repaid, so we use older loans. Disparate Impact only needs approval rates, so it can also be checked every week.

**Trigger for re-audit:**  
If the **TPR or FPR gap** goes above 0.05 for two months in a row, or **Disparate Impact** falls below 0.8, the model is checked again and the fix is updated.

**Why continuous monitoring is needed even after debiasing:**  
The fix were built using old data. Over time applicants, incomes and the economy all change, so the fix may stop working. SDAIA's Deploy and Monitor stage also says the model should be checked regularly and changed if problems appear again. 

---
# Results Summary Dashboard

Run the cell below **after completing all calculations** in Parts 1 and 2.  
It will print a side-by-side comparison of all computed metrics for both scenarios.


In [12]:
# ── Summary table — run after completing Q1.1 and Q2.1 ──────────────────────
# (This cell will raise a NameError if you have not yet filled in the TODO blocks above.)

rows = [
    ("Statistical Parity",    stat_parity,  stat_parity2,  "≈ 0",  lambda v: abs(v) < 0.05),
    ("Disparate Impact",       disparate_impact, disparate_impact2, "≥ 0.8", lambda v: v >= 0.8),
    ("Equal Opportunity",      eq_opp,       eq_opp2,       "≈ 0",  lambda v: abs(v) < 0.05),
    ("TPR gap (Eq. Odds)",     eq_opp,       eq_opp2,       "≈ 0",  lambda v: abs(v) < 0.05),
    ("FPR gap (Eq. Odds)",     fpr_gap,      fpr_gap2,      "≈ 0",  lambda v: abs(v) < 0.05),
]

def colour(val, check):
    ok = check(val)
    c  = "#15803D" if ok else "#DC2626"
    sym = "✓" if ok else "✗"
    return f"<span style='color:{c}'><b>{val:.4f} {sym}</b></span>"

header = (
    "<table style='border-collapse:collapse; font-family:monospace; width:100%; font-size:13px;'>"
    "<tr style='background:#1B3A6B; color:white;'>"
    "<th style='padding:8px 12px; text-align:left;'>Metric</th>"
    "<th style='padding:8px 12px; text-align:center;'>Target</th>"
    "<th style='padding:8px 12px; text-align:center;'>Part 1 · Hiring</th>"
    "<th style='padding:8px 12px; text-align:center;'>Part 2 · Loan</th>"
    "</tr>"
)
body_html = ""
for i, (name, v1, v2, target, chk) in enumerate(rows):
    bg = "#F3F6FB" if i % 2 == 0 else "#FFFFFF"
    body_html += (
        f"<tr style='background:{bg};'>"
        f"<td style='padding:7px 12px;'>{name}</td>"
        f"<td style='padding:7px 12px; text-align:center; color:#6B7280;'>{target}</td>"
        f"<td style='padding:7px 12px; text-align:center;'>{colour(v1, chk)}</td>"
        f"<td style='padding:7px 12px; text-align:center;'>{colour(v2, chk)}</td>"
        "</tr>"
    )
footer = "</table>"
display(HTML(header + body_html + footer))
print("\n✅ Summary generated. Green = criterion met  |  Red = criterion violated")


Metric,Target,Part 1 · Hiring,Part 2 · Loan
Statistical Parity,≈ 0,-0.1000 ✗,-0.1000 ✗
Disparate Impact,≥ 0.8,0.7500 ✗,0.8333 ✓
Equal Opportunity,≈ 0,-0.3000 ✗,-0.2119 ✗
TPR gap (Eq. Odds),≈ 0,-0.3000 ✗,-0.2119 ✗
FPR gap (Eq. Odds),≈ 0,0.0500 ✓,0.0778 ✗



✅ Summary generated. Green = criterion met  |  Red = criterion violated


---
## Submission Checklist

Before submitting, confirm each item below by replacing `[ ]` with `[x]`.

- [✅] Student names and IDs filled in at the top of this notebook
- [✅] All `...` placeholders replaced with actual code in Q1.1 (all four metrics)
- [✅] All `...` placeholders replaced with actual code in Q2.1 (all four metrics)
- [✅] Written answers completed for Q1.2, Q1.3, Q2.2, Q3.1, Q3.2, Q3.3
- [✅] Summary dashboard cell runs without errors
- [✅] Notebook saved with **Kernel → Restart & Run All** to verify clean execution
- [✅] File saved as `.ipynb` and submitted through the course portal

---
<div style="text-align:center; font-family:Arial,sans-serif; color:#9CA3AF; font-size:12px; margin-top: 12px;">
ARTI 459 &nbsp;•&nbsp; AI Ethics &nbsp;•&nbsp; Group Assignment — AI Fairness Metrics
</div>
